In [2]:
from pathlib import Path
import pandas as pd

from recompute_metrics_for_robustness import find_project_root, run_scenario

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

project_root = find_project_root(Path.cwd().resolve())
project_root

PosixPath('/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans')

In [3]:
pangram_result = run_scenario(project_root, label='pangram', exclude_ids=[5540, 9418])
pangram_ranks = pd.read_csv(pangram_result['ranks_path'])

In [4]:
roberta_result = run_scenario(
    project_root,
    label='roberta',
    exclude_ids=[5540, 10499, 7195, 3761, 1214, 5047, 3891],
)
roberta_ranks = pd.read_csv(roberta_result['ranks_path'])

In [5]:
# how many ns left
base_n = 60
pangram_n = base_n - len(pangram_result['exclude_ids'])
roberta_n = base_n - len(roberta_result['exclude_ids'])

keys = ['file', 'metric', 'prompt', 'model']
base = pangram_ranks[keys + ['baseline', 'baseline_rank']].rename(
    columns={'baseline': f'{base_n}_value', 'baseline_rank': f'{base_n}_rank'}
)
pang = pangram_ranks[keys + ['filtered', 'filtered_rank', 'rank_shift']].rename(
    columns={
        'filtered': f'{pangram_n}_value',
        'filtered_rank': f'{pangram_n}_rank',
        'rank_shift': f'{pangram_n}_rank_shift',
    },
)
rob = roberta_ranks[keys + ['filtered', 'filtered_rank', 'rank_shift']].rename(
    columns={
        'filtered': f'{roberta_n}_value',
        'filtered_rank': f'{roberta_n}_rank',
        'rank_shift': f'{roberta_n}_rank_shift',
    },
)

combined = (
    base.merge(pang, on=keys, how='left')
    .merge(rob, on=keys, how='left')
    .assign(prompt_order=lambda d: d['prompt'].astype(str).map(lambda p: {'original': 0, 'large': 1}.get(p, 999)))
    .sort_values(['prompt_order', 'file', 'metric', f'{base_n}_rank'])
    .reset_index(drop=True)
)

display_cols = [
    'model',
    f'{base_n}_value', f'{base_n}_rank',
    f'{pangram_n}_value', f'{pangram_n}_rank', f'{pangram_n}_rank_shift',
    f'{roberta_n}_value', f'{roberta_n}_rank', f'{roberta_n}_rank_shift',
]

In [6]:
tables_by_prompt = {}
for prompt_name in ['original', 'large'] + sorted(
    p for p in combined['prompt'].astype(str).unique() if p not in {'original', 'large'}
):
    prompt_df = combined[combined['prompt'].astype(str) == prompt_name].copy()
    if prompt_df.empty:
        continue
    prompt_tables = {}
    for (file_name, metric_name), g in prompt_df.groupby(['file', 'metric'], sort=True):
        prompt_tables[(file_name, metric_name)] = g[display_cols].sort_values(f'{base_n}_rank').reset_index(drop=True)
    tables_by_prompt[prompt_name] = prompt_tables

In [7]:
print('Prompt: original')
for (file_name, metric_name), table in tables_by_prompt.get('original', {}).items():
    print(f'metric: {metric_name} | file: {file_name}')
    display(table)

Prompt: original
metric: char_coherence_mean | file: character/character_metrics_agg_per_character_zero_policy.csv


,model,60_value,60_rank,58_value,58_rank,58_rank_shift,53_value,53_rank,53_rank_shift
0,human,0.462146,1,0.470473,1,0,0.473443,1,0
1,internvl3,0.399559,2,0.403112,2,0,0.392668,3,1
2,gpt4o,0.389699,3,0.397290,3,0,0.395630,2,-1
3,claude45,0.342457,4,0.349921,4,0,0.337401,4,0
4,qwen3vl,0.338203,5,0.341993,5,0,0.331086,5,0
5,llama4scout,0.161630,6,0.165722,6,0,0.150101,6,0


metric: coref_ratio_mean | file: coreference/coref_metrics_agg.csv


,model,60_value,60_rank,58_value,58_rank,58_rank_shift,53_value,53_rank,53_rank_shift
0,human,0.773725,1,0.775713,1,0,0.784032,1,0
1,llama4scout,0.730055,2,0.725786,2,0,0.734633,2,0
2,qwen3vl,0.703818,3,0.700294,3,0,0.701569,3,0
3,gpt4o,0.647946,4,0.641536,4,0,0.643829,4,0
4,internvl3,0.631249,5,0.633339,5,0,0.631506,5,0
5,claude45,0.582792,6,0.586089,6,0,0.600347,6,0


metric: groovist_mean | file: groovist/groovist_metrics_agg.csv


...

,model,60_value,60_rank,58_value,58_rank,58_rank_shift,53_value,53_rank,53_rank_shift
0,human,0.500014,1,0.501179,1,0,0.501380,1,0
1,gpt4o,0.455157,2,0.458491,2,0,0.456824,2,0
2,qwen3vl,0.448251,3,0.450185,3,0,0.450670,3,0
3,internvl3,0.432624,4,0.437233,4,0,0.434859,4,0
4,claude45,0.412418,5,0.414814,5,0,0.423653,5,0
5,llama4scout,0.322128,6,0.321727,6,0,0.322534,6,0


metric: ncs_geometric_mean | file: ncs/ncs_aggregate_zero_policy.csv


,model,60_value,60_rank,58_value,58_rank,58_rank_shift,53_value,53_rank,53_rank_shift
0,human,0.363801,1,0.366430,1,0,0.368165,1,0
1,gpt4o,0.317019,2,0.321558,2,0,0.316488,3,1
2,qwen3vl,0.316177,3,0.319907,3,0,0.320888,2,-1
3,internvl3,0.304617,4,0.308566,4,0,0.300936,4,0
4,claude45,0.272528,5,0.274347,5,0,0.291999,5,0
5,llama4scout,0.059710,6,0.061754,6,0,0.057677,6,0


metric: topic_switch_rate_mean | file: topic_modelling/topic_switch_metrics_agg.csv


,model,60_value,60_rank,58_value,58_rank,58_rank_shift,53_value,53_rank,53_rank_shift
0,human,0.425266,1,0.422535,1,0,0.416692,1,0
1,qwen3vl,0.384454,2,0.379872,2,0,0.387460,2,0
2,gpt4o,0.282886,3,0.285702,3,0,0.270843,4,1
3,claude45,0.271575,4,0.270963,5,1,0.291183,3,-1
4,internvl3,0.270864,5,0.273471,4,-1,0.266420,5,0
5,llama4scout,0.258780,6,0.255536,6,0,0.253446,6,0


In [8]:
print('Prompt: large')
for (file_name, metric_name), table in tables_by_prompt.get('large', {}).items():
    print(f'metric: {metric_name} | file: {file_name}')
    display(table)

Prompt: large
metric: char_coherence_mean | file: character/character_metrics_agg_per_character_zero_policy.csv


,model,60_value,60_rank,58_value,58_rank,58_rank_shift,53_value,53_rank,53_rank_shift
0,human,0.414627,1,0.413952,1,0,0.418767,1,0
1,gpt4o,0.381980,2,0.386111,2,0,0.388262,2,0
2,qwen3vl,0.321560,3,0.327902,3,0,0.338527,3,0
3,internvl3,0.317286,4,0.316270,5,1,0.300533,5,1
4,claude45,0.308233,5,0.317238,4,-1,0.306953,4,-1
5,llama4scout,0.285095,6,0.290631,6,0,0.274884,6,0


metric: coref_ratio_mean | file: coreference/coref_metrics_agg.csv


,model,60_value,60_rank,58_value,58_rank,58_rank_shift,53_value,53_rank,53_rank_shift
0,human,0.678780,1,0.677865,1,0,0.683982,1,0
1,gpt4o,0.660918,2,0.663064,2,0,0.662158,2,0
2,llama4scout,0.633780,3,0.639640,3,0,0.631999,3,0
3,qwen3vl,0.622731,4,0.625173,4,0,0.621136,4,0
4,claude45,0.589139,5,0.592352,5,0,0.590547,5,0
5,internvl3,0.560120,6,0.557987,6,0,0.561675,6,0


metric: groovist_mean | file: groovist/groovist_metrics_agg.csv


...

,model,60_value,60_rank,58_value,58_rank,58_rank_shift,53_value,53_rank,53_rank_shift
0,human,0.466622,1,0.468646,1,0,0.476450,1,0
1,gpt4o,0.456155,2,0.460062,2,0,0.461994,2,0
2,qwen3vl,0.411674,3,0.414846,3,0,0.413900,3,0
3,internvl3,0.405709,4,0.408645,4,0,0.403501,5,1
4,claude45,0.403739,5,0.405416,5,0,0.410229,4,-1
5,llama4scout,0.378052,6,0.383542,6,0,0.378803,6,0


metric: ncs_geometric_mean | file: ncs/ncs_aggregate_zero_policy.csv


,model,60_value,60_rank,58_value,58_rank,58_rank_shift,53_value,53_rank,53_rank_shift
0,human,0.364572,1,0.367845,1,0,0.372893,1,0
1,gpt4o,0.319378,2,0.322234,2,0,0.329252,2,0
2,qwen3vl,0.272783,3,0.275556,3,0,0.275019,3,0
3,internvl3,0.271808,4,0.274409,4,0,0.263823,5,1
4,claude45,0.259843,5,0.262658,5,0,0.266628,4,-1
5,llama4scout,0.171512,6,0.177416,6,0,0.163128,6,0


metric: topic_switch_rate_mean | file: topic_modelling/topic_switch_metrics_agg.csv


,model,60_value,60_rank,58_value,58_rank,58_rank_shift,53_value,53_rank,53_rank_shift
0,llama4scout,0.351192,1,0.357500,1,0,0.355632,1,0
1,human,0.334410,2,0.334702,2,0,0.339978,2,0
2,gpt4o,0.314267,3,0.318131,3,0,0.312049,3,0
3,internvl3,0.299822,4,0.306404,4,0,0.295466,4,0
4,qwen3vl,0.275841,5,0.278660,5,0,0.264107,6,1
5,claude45,0.261740,6,0.260739,6,0,0.271550,5,-1
